In [1]:
# Imports nécessaires
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import re
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration de l'affichage
%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [2]:
# Création de la session Spark
spark = SparkSession.builder \
    .appName("Apache Log Analysis - Local Mode") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

# Réduire la verbosité des logs
spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")
print(f"Spark master: {spark.sparkContext.master}")

Spark version: 3.5.0
Spark master: local[*]


In [5]:
hdfs_path = "../data/access.log"  

logs_raw = spark.read.text(hdfs_path)

logs_raw.count()

3602

In [6]:
logs_raw.show(5, truncate=False)


+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                                                 |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|57.70.252.24 - - [01/Nov/2024:00:12:53 +0100] "GET /products/P001 HTTP/1.1" 404 76 "-" "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0 Safari/537.36"      |
|168.255.202.227 - - [01/Nov/2024:00:13:40 +0100] "GET /faq HTTP/1.1" 200 19721 "-" "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0 Safari/537.36"          |
|109.16.15.24 - - [01/Nov/2024:01:20:27 +0100] "DELETE /contact HTTP/1.1" 2

In [7]:
log_pattern = r'^(\S+) \S+ \S+ \[(.*?)\] "(\S+) (.*?) (HTTP/\S+)" (\d{3}) (\S+) "(.*?)" "(.*?)"'

logs_df = logs_raw.select(
    regexp_extract(col("value"), log_pattern, 1).alias("ip"),
    regexp_extract(col("value"), log_pattern, 2).alias("timestamp"),
    regexp_extract(col("value"), log_pattern, 3).alias("method"),
    regexp_extract(col("value"), log_pattern, 4).alias("url"),
    regexp_extract(col("value"), log_pattern, 6).cast("int").alias("status_code")
).filter(col("ip") != "")

logs_df.cache()

logs_df.printSchema()
logs_df.show(5, truncate=120)

root
 |-- ip: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- method: string (nullable = true)
 |-- url: string (nullable = true)
 |-- status_code: integer (nullable = true)

+---------------+--------------------------+------+--------------+-----------+
|             ip|                 timestamp|method|           url|status_code|
+---------------+--------------------------+------+--------------+-----------+
|   57.70.252.24|01/Nov/2024:00:12:53 +0100|   GET|/products/P001|        404|
|168.255.202.227|01/Nov/2024:00:13:40 +0100|   GET|          /faq|        200|
|   109.16.15.24|01/Nov/2024:01:20:27 +0100|DELETE|      /contact|        200|
| 161.81.216.153|01/Nov/2024:01:35:19 +0100|   GET|    /blog/news|        200|
|  133.231.61.64|01/Nov/2024:01:37:51 +0100|   GET|    /blog/news|        404|
+---------------+--------------------------+------+--------------+-----------+
only showing top 5 rows



In [10]:
logs_df = logs_df \
    .withColumn(
        "status_group",
        when(col("status_code").between(200, 299), "2xx")
        .when(col("status_code").between(400, 499), "4xx")
        .when(col("status_code").between(500, 599), "5xx")
        .otherwise("other")
    ) \
    .withColumn(
        "is_error",
        col("status_code") >= 400
    ) \
    .withColumn(
        "timestamp_parsed",
        to_timestamp(col("timestamp"), "dd/MMM/yyyy:HH:mm:ss Z")
    ) \
    .withColumn(
        "hour",
        hour(col("timestamp_parsed"))
    )

logs_df.select("status_code", "status_group", "is_error", "hour").show(5)


+-----------+------------+--------+----+
|status_code|status_group|is_error|hour|
+-----------+------------+--------+----+
|        404|         4xx|    true|  23|
|        200|         2xx|   false|  23|
|        200|         2xx|   false|   0|
|        200|         2xx|   false|   0|
|        404|         4xx|    true|   0|
+-----------+------------+--------+----+
only showing top 5 rows



In [11]:
profil_ip_df = logs_df.groupBy("ip").agg(
    
    count("*").alias("nb_requetes"),
    
    sum(when(col("is_error") == True, 1).otherwise(0)).alias("nb_erreurs"),
    
    sum(
        when(col("hour").between(1, 4), 1).otherwise(0)
    ).alias("nb_requetes_nuit")
)

profil_ip_df = profil_ip_df.withColumn(
    "taux_erreur",
    round(col("nb_erreurs") / col("nb_requetes"), 3)
)

profil_ip_df = profil_ip_df.select(
    "ip",
    "nb_requetes",
    "nb_erreurs",
    "taux_erreur",
    "nb_requetes_nuit"
)


profil_ip_df.show(5, truncate=False)

+---------------+-----------+----------+-----------+----------------+
|ip             |nb_requetes|nb_erreurs|taux_erreur|nb_requetes_nuit|
+---------------+-----------+----------+-----------+----------------+
|224.216.108.238|34         |12        |0.353      |10              |
|151.142.3.195  |23         |4         |0.174      |5               |
|165.174.57.76  |21         |7         |0.333      |2               |
|50.35.23.170   |41         |6         |0.146      |5               |
|194.24.56.40   |42         |6         |0.143      |8               |
+---------------+-----------+----------+-----------+----------------+
only showing top 5 rows

